# Species-Only PseudoPA Training Walkthrough

This notebook prepares plot-level `species_set` tables directly from the downloaded GeoPlant files in `dataset/data/`.

It uses the prepared `geoplant_pseudopa` baseline for:

- PA masked reconstruction
- PO positive-unlabeled ranking
- joint PA+PO training
- prevalence-aware bias initialization and negative sampling

It intentionally stays species-only and does not use GPS or external covariates.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.neighbors import BallTree
from sklearn.neighbors import KNeighborsClassifier
from torch.utils.data import DataLoader

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from geoplant_pseudopa import (
    JointTrainingConfig,
    SpeciesSetDataset,
    evaluate_completion_from_inputs,
    evaluate_completion_from_pa,
    evaluate_pa_topk,
    train_joint_model,
)


/Users/lukaspicek/Documents/Projects/INRIA/GLC/GeoPlant/.venv-pseudopa/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RAW_PA_TRAIN = REPO_ROOT / "dataset" / "data" / "PresenceAbsenceSurveys" / "PA_metadata_train.csv"
RAW_PA_TEST = REPO_ROOT / "dataset" / "data" / "PresenceAbsenceSurveys" / "PA_metadata_test.csv"
RAW_PA_TEST_LABELS = REPO_ROOT / "dataset" / "data" / "PresenceAbsenceSurveys" / "test_labels.csv"
RAW_PO_TRAIN = REPO_ROOT / "dataset" / "data" / "PresenceOnlyOccurrences" / "PO_metadata_train.csv"
PREPARED_DIR = NOTEBOOK_DIR / "outputs" / "prepared_metadata"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

# Example: None, ["Denmark"], ["France", "Switzerland"]
COUNTRY_FILTER = None
TEST_COUNTRY_FILTER = ["Denmark"]
PO_COUNTRY_REFERENCE_SAMPLES = 200000
PO_COUNTRY_K = 5
PO_NEIGHBOR_RADIUS_KM = 25.0
PO_NEIGHBOR_MIN_SURVEYS = 5
PO_NEIGHBOR_MAX_SURVEYS = 100

VAL_FRACTION = 0.1
SPLIT_SEED = 42
NUM_SPECIES = 15286
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64

config = JointTrainingConfig(
    num_species=NUM_SPECIES,
    batch_size=BATCH_SIZE,
    warmup_pa_epochs=5,
    joint_epochs=20,
    use_location=True,
    mask_ratio=0.6,
    pa_negative_ratio=8.0,
    po_negative_ratio=4.0,
    lambda_po=1.0,
)

RAW_PA_TRAIN, RAW_PO_TRAIN, COUNTRY_FILTER, TEST_COUNTRY_FILTER, DEVICE

(PosixPath('/Users/lukaspicek/Documents/Projects/INRIA/GLC/GeoPlant/dataset/data/PresenceAbsenceSurveys/PA_metadata_train.csv'),
 PosixPath('/Users/lukaspicek/Documents/Projects/INRIA/GLC/GeoPlant/dataset/data/PresenceOnlyOccurrences/PO_metadata_train.csv'),
 None,
 ['Denmark'],
 'cpu')

In [3]:
def aggregate_observations(raw_frame: pd.DataFrame, source_name: str) -> pd.DataFrame:
    required_columns = {"surveyId", "speciesId"}
    missing = required_columns - set(raw_frame.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    aggregated = (
        raw_frame.groupby("surveyId")
        .agg(
            species_set=("speciesId", lambda col: "{" + ",".join(str(int(species_id)) for species_id in sorted(set(col.dropna().astype(int)))) + "}"),
            lat=("lat", "mean"),
            lon=("lon", "mean"),
        )
        .reset_index()
        .rename(columns={"surveyId": "survey_id"})
    )
    aggregated["source"] = source_name
    aggregated["survey_id"] = aggregated["survey_id"].astype(str)
    return aggregated


def apply_country_filter(raw_frame: pd.DataFrame, country_filter):
    if country_filter is None:
        return raw_frame.reset_index(drop=True)
    if "country" in raw_frame.columns:
        return raw_frame[raw_frame["country"].isin(country_filter)].reset_index(drop=True)
    return raw_frame.reset_index(drop=True)


def infer_po_country_from_pa(pa_reference: pd.DataFrame, po_frame: pd.DataFrame) -> pd.DataFrame:
    required_pa = {"lat", "lon", "country"}
    required_po = {"lat", "lon"}
    if not required_pa.issubset(pa_reference.columns):
        raise ValueError(f"PA reference is missing columns: {sorted(required_pa - set(pa_reference.columns))}")
    if not required_po.issubset(po_frame.columns):
        raise ValueError(f"PO frame is missing columns: {sorted(required_po - set(po_frame.columns))}")

    reference = pa_reference[["lat", "lon", "country"]].dropna().copy()
    if len(reference) > PO_COUNTRY_REFERENCE_SAMPLES:
        reference = reference.sample(PO_COUNTRY_REFERENCE_SAMPLES, random_state=SPLIT_SEED)

    knn = KNeighborsClassifier(n_neighbors=PO_COUNTRY_K, weights="distance")
    knn.fit(reference[["lat", "lon"]], reference["country"])

    predicted = po_frame.copy()
    predicted["country"] = knn.predict(predicted[["lat", "lon"]])
    return predicted


raw_pa_reference = pd.read_csv(RAW_PA_TRAIN)
raw_pa_train = apply_country_filter(raw_pa_reference, COUNTRY_FILTER)
raw_po_train = pd.read_csv(RAW_PO_TRAIN)
if COUNTRY_FILTER is not None:
    raw_po_train = infer_po_country_from_pa(raw_pa_reference, raw_po_train)
    raw_po_train = apply_country_filter(raw_po_train, COUNTRY_FILTER)

prepared_pa = aggregate_observations(raw_pa_train, "PA")
prepared_po = aggregate_observations(raw_po_train, "PO")

unique_surveys = prepared_pa["survey_id"].sample(frac=1.0, random_state=SPLIT_SEED).reset_index(drop=True)
num_val = max(1, int(round(len(unique_surveys) * VAL_FRACTION)))
val_surveys = set(unique_surveys.iloc[:num_val])

prepared_pa["subset"] = prepared_pa["survey_id"].apply(lambda survey_id: "val" if survey_id in val_surveys else "train")
prepared_po["subset"] = "train"

pa_train_frame = prepared_pa[prepared_pa["subset"] == "train"].reset_index(drop=True)
pa_val_frame = prepared_pa[prepared_pa["subset"] == "val"].reset_index(drop=True)
po_train_frame = prepared_po.reset_index(drop=True)

prepared_train_path = PREPARED_DIR / "pa_species_set_train.csv"
prepared_val_path = PREPARED_DIR / "pa_species_set_val.csv"
prepared_po_path = PREPARED_DIR / "po_species_set_train.csv"
pa_train_frame.to_csv(prepared_train_path, index=False)
pa_val_frame.to_csv(prepared_val_path, index=False)
po_train_frame.to_csv(prepared_po_path, index=False)

pd.DataFrame(
    [
        {"split": "PA train", "plots": len(pa_train_frame), "countries": COUNTRY_FILTER, "path": str(prepared_train_path)},
        {"split": "PO train", "plots": len(po_train_frame), "countries": COUNTRY_FILTER, "path": str(prepared_po_path)},
        {"split": "PA val", "plots": len(pa_val_frame), "countries": COUNTRY_FILTER, "path": str(prepared_val_path)},
    ]
)

,split,plots,countries,path
0,PA train,80088,None,/Users/lukaspicek/Documents/Projects/INRIA/GLC...
1,PO train,3845533,None,/Users/lukaspicek/Documents/Projects/INRIA/GLC...
2,PA val,8899,None,/Users/lukaspicek/Documents/Projects/INRIA/GLC...


In [4]:
pa_train_dataset = SpeciesSetDataset(pa_train_frame, config.num_species)
po_train_dataset = SpeciesSetDataset(po_train_frame, config.num_species)
pa_val_dataset = SpeciesSetDataset(pa_val_frame, config.num_species)

pa_train_loader = DataLoader(pa_train_dataset, batch_size=config.batch_size, shuffle=True)
po_train_loader = DataLoader(po_train_dataset, batch_size=config.batch_size, shuffle=len(po_train_dataset) > 0)
pa_val_loader = DataLoader(pa_val_dataset, batch_size=config.batch_size, shuffle=False)

example_batch = next(iter(pa_train_loader))
print("PA batch shape:", tuple(example_batch["x"].shape))
print("Mean species per PA plot in first batch:", float(example_batch["x"].sum(dim=1).float().mean()))
print("PO training plots available:", len(po_train_dataset))
if len(po_train_dataset) == 0 and COUNTRY_FILTER is not None:
    print("PO is empty for the current COUNTRY_FILTER because the PO file has no country column. Training will continue with PA only.")

PA batch shape: (64, 15286)
Mean species per PA plot in first batch: 15.625
PO training plots available: 3845533


## Train The Model

The training loop runs a short PA-only warmup and then switches to joint PA+PO optimization.

The default `mask_ratio=0.6` drops more PA species during training than before, which pushes the model harder toward reconstructing missing community members from partial observations.

In [5]:
model, history = train_joint_model(
    pa_train_loader,
    po_train_loader,
    pa_val_loader,
    config=config,
    device=DEVICE,
)

history_df = pd.DataFrame(history)
history_df

Starting training with {'device': 'cpu', 'warmup_pa_epochs': 5, 'joint_epochs': 20, 'pa_train_plots': 80088, 'po_train_plots': 3845533, 'val_plots': 8899, 'batch_size': 64}
[Warmup 1/5]


  train: {'pa_loss': 0.2517, 'po_loss': 0.0, 'mask_fraction': 0.6017}
  val: {'recall@5': 0.2084, 'precision@5': 0.5079, 'recall@10': 0.3245, 'precision@10': 0.4113, 'recall@20': 0.4624, 'precision@20': 0.3048}
[Warmup 2/5]


KeyboardInterrupt: 

In [ ]:
metric_columns = [
    "epoch",
    "stage",
    "pa_loss",
    "po_loss",
    "mask_fraction",
    "recall@5",
    "recall@10",
    "recall@20",
    "precision@5",
    "precision@10",
    "precision@20",
]
history_df[metric_columns]

In [ ]:
final_val_metrics = evaluate_pa_topk(model, pa_val_loader, device=torch.device(DEVICE))
pd.Series(final_val_metrics, name="value")

## Denmark Composition Completion Test

This builds a labeled PA test set by combining `PA_metadata_test.csv` with `test_labels.csv`.

By default it evaluates only Denmark test plots via `TEST_COUNTRY_FILTER = ["Denmark"]`.

The completion benchmark hides part of each full PA composition, uses the remaining species as a PO-like partial input, and measures how well the model recovers the hidden community members.

Use this as a held-out report, not for model selection during iteration.

In [ ]:
raw_pa_test = apply_country_filter(pd.read_csv(RAW_PA_TEST), TEST_COUNTRY_FILTER)
test_labels = pd.read_csv(RAW_PA_TEST_LABELS)
test_labels = test_labels.rename(columns={"surveyId": "survey_id", "predictions": "species_set"})
test_labels["survey_id"] = test_labels["survey_id"].astype(str)
test_labels["species_set"] = test_labels["species_set"].apply(
    lambda value: "{" + ",".join(token for token in str(value).split() if token) + "}"
)

test_frame = raw_pa_test.rename(columns={"surveyId": "survey_id"}).copy()
test_frame["survey_id"] = test_frame["survey_id"].astype(str)
test_frame = test_frame.merge(test_labels[["survey_id", "species_set"]], on="survey_id", how="inner")
test_frame = test_frame[["survey_id", "lat", "lon", "species_set"]].drop_duplicates().reset_index(drop=True)
test_frame["source"] = "PA"
test_frame["subset"] = "test"

prepared_test_path = PREPARED_DIR / "pa_species_set_test.csv"
test_frame.to_csv(prepared_test_path, index=False)

test_dataset = SpeciesSetDataset(test_frame, config.num_species)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

pd.DataFrame([
    {"split": "PA test", "plots": len(test_frame), "countries": TEST_COUNTRY_FILTER, "path": str(prepared_test_path)}
])

In [ ]:
completion_results = []
for drop_rate in (0.3, 0.5, 0.7):
    completion_results.append(
        evaluate_completion_from_pa(
            model,
            test_loader,
            device=torch.device(DEVICE),
            drop_rate=drop_rate,
            ks=(5, 10, 20),
            seed=SPLIT_SEED,
        )
    )

pd.DataFrame(completion_results)

## Denmark PA Test From Real Surrounding PO

This is closer to your current hypothesis: for each Denmark PA test plot, gather nearby PO surveys in geographic space, union their observed species, and use that PO neighborhood composition as the model input.

The target remains the full Denmark PA test composition.

In [ ]:
def build_species_vector(species_ids, num_species):
    vector = torch.zeros(num_species, dtype=torch.float32)
    if species_ids:
        vector[list(sorted(species_ids))] = 1.0
    return vector


po_test_source = pd.read_csv(RAW_PO_TRAIN)
if TEST_COUNTRY_FILTER is not None:
    po_test_source = infer_po_country_from_pa(raw_pa_reference, po_test_source)
    po_test_source = apply_country_filter(po_test_source, TEST_COUNTRY_FILTER)

po_test_surveys = (
    po_test_source.groupby('surveyId')
    .agg(
        lat=('lat', 'mean'),
        lon=('lon', 'mean'),
        species_set=('speciesId', lambda col: set(int(species_id) for species_id in col.dropna().astype(int))),
    )
    .reset_index()
    .rename(columns={'surveyId': 'survey_id'})
)

po_coords_rad = np.deg2rad(po_test_surveys[['lat', 'lon']].to_numpy())
po_tree = BallTree(po_coords_rad, metric='haversine')
radius_rad = PO_NEIGHBOR_RADIUS_KM / 6371.0

test_eval_frame = test_frame.merge(
    raw_pa_test.rename(columns={'surveyId': 'survey_id'})[['survey_id', 'lat', 'lon']],
    on='survey_id',
    how='left',
)

input_vectors = []
target_vectors = []
neighborhood_rows = []

for row in test_eval_frame.itertuples(index=False):
    query = np.deg2rad([[float(row.lat), float(row.lon)]])
    within_radius = po_tree.query_radius(query, r=radius_rad)[0].tolist()

    if len(within_radius) < PO_NEIGHBOR_MIN_SURVEYS:
        _, knn_idx = po_tree.query(query, k=min(PO_NEIGHBOR_MIN_SURVEYS, len(po_test_surveys)))
        neighbor_idx = knn_idx[0].tolist()
    else:
        neighbor_idx = within_radius[:PO_NEIGHBOR_MAX_SURVEYS]

    neighbor_species = set()
    for idx in neighbor_idx:
        neighbor_species.update(po_test_surveys.iloc[idx]['species_set'])

    target_species = set(int(token) for token in str(row.species_set).strip('{}').split(',') if token)
    input_vectors.append(build_species_vector(neighbor_species, config.num_species))
    target_vectors.append(build_species_vector(target_species, config.num_species))
    neighborhood_rows.append(
        {
            'survey_id': row.survey_id,
            'po_neighbor_surveys': len(neighbor_idx),
            'po_neighbor_species': len(neighbor_species),
            'pa_species': len(target_species),
        }
    )

neighbor_summary = pd.DataFrame(neighborhood_rows)
neighbor_summary.describe(include='all')

In [ ]:
po_neighbor_input = torch.stack(input_vectors)
pa_test_target = torch.stack(target_vectors)

test_locations = torch.tensor(test_eval_frame[["lat", "lon"]].to_numpy(dtype=float), dtype=torch.float32)

po_neighbor_metrics = evaluate_completion_from_inputs(
    model,
    x_input=po_neighbor_input,
    x_true=pa_test_target,
    input_loc=test_locations,
    device=torch.device(DEVICE),
    ks=(5, 10, 20),
)

pd.Series(po_neighbor_metrics, name='value')

In [ ]:
# Optional: save the trained model and training history.
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

torch.save(
    {
        "state_dict": model.state_dict(),
        "config": config.__dict__,
        "history": history,
        "prepared_pa_train": str(prepared_train_path),
        "prepared_pa_val": str(prepared_val_path),
        "prepared_po_train": str(prepared_po_path),
    },
    OUTPUT_DIR / "species_only_joint_model.pt",
)

history_df.to_csv(OUTPUT_DIR / "species_only_joint_history.csv", index=False)
OUTPUT_DIR